In [1]:
# !pip install tensorflow==2.16.1
import json
import os
import sys
import asyncio
import argparse
from collections import defaultdict
import time
import re

os.environ["JAX_PLATFORMS"] = "cpu"

import torch
import numpy as np
import jax.numpy as jnp
import jax
import orbax
import orbax.checkpoint as ocp
from etils import epath
from jax.sharding import PartitionSpec as PS
from flax.traverse_util import flatten_dict, unflatten_dict
import base64


def decode_base64(encoded_str):
    decoded_bytes = base64.b64decode(encoded_str)
    decoded_str = decoded_bytes.decode('utf-8')
    return decoded_str

def encode_base64(decoded_str):
    # decoded_str = "opt_state.mu.params.token_embedder.embedding"
    encoded_string = base64.b64encode(decoded_str.encode('utf-8')).decode('utf-8')
    return encoded_string


In [2]:
# load
mesh_axes = ['data', 'stage', 'fsdp', 'fsdp_transpose', 'sequence', 'tensor', 'tensor_transpose', 'tensor_sequence', 'expert', 'autoregressive']
devices = np.asarray(jax.devices()).reshape([1] * len(mesh_axes))
mesh = jax.sharding.Mesh(devices, mesh_axes)
sharding = jax.sharding.NamedSharding(mesh, PS()) # Sharding is None because we use cpu to load weights
weight_dtype = jnp.bfloat16 # set restore weights dtype, np.float32 or np.float16
abstract_unboxed_params = {}
struct_json = 'gs://newproject-1-llm_base_models_europe-west4/v3.5mini/DreamMiniXL0506/v3.5mini_params_shape.json'
with epath.Path(struct_json).open('r') as f:
    struct_json = json.load(f)
    
for k, shape in struct_json.items():
    if 'dense_proj2.bias' in k:  # ；这个地方千万注意，因为这个参数的名字叫dense_proj2.bias不是 bias
        k = k.split('.')[:-2] + ['dense_proj2.bias']
    else:
        k = k.split('.')
    print(k, shape)
    abstract_unboxed_params[tuple(k)] = jax.ShapeDtypeStruct(shape=shape, dtype=weight_dtype, sharding=sharding)    
    
abstract_unboxed_params = unflatten_dict(abstract_unboxed_params)

checkpoint_dir = 'gs://newproject-1-llm_base_models_europe-west4/v3.5mini/DreamMiniXL0506/checkpoints/180500/items'
ckpt = epath.Path(checkpoint_dir)
ckptr = ocp.PyTreeCheckpointer()
restore_args = ocp.checkpoint_utils.construct_restore_args(abstract_unboxed_params)
# 如果restored只是一个带有模型名字的字典，没有具体的value矩阵，可以检查下abstract_unboxed_params是不是多了或者漏了params这个key
restored = ckptr.restore(
  ckpt, item={'params': abstract_unboxed_params}, transforms={}, restore_args={'params': restore_args},
)

2025-06-03 07:40:54.278576: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1748936454.292156  122192 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748936454.296078  122192 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1748936454.308289  122192 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1748936454.308304  122192 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1748936454.308306  122192 computation_placer.cc:177] computation placer alr

['params', 'decoder', 'compose_0', 'mudd_postnorm_0', 'scale'] [2048]
['params', 'decoder', 'compose_0', 'mudd_prenorm_0', 'scale'] [2048]
['params', 'decoder', 'compose_1', 'mudd_postnorm_1', 'scale'] [2048]
['params', 'decoder', 'compose_1', 'mudd_prenorm_1', 'scale'] [2048]
['params', 'decoder', 'compose_10', 'mudd_postnorm_10', 'scale'] [2048]
['params', 'decoder', 'compose_10', 'mudd_prenorm_10', 'scale'] [2048]
['params', 'decoder', 'compose_11', 'mudd_postnorm_11', 'scale'] [2048]
['params', 'decoder', 'compose_11', 'mudd_prenorm_11', 'scale'] [2048]
['params', 'decoder', 'compose_12', 'mudd_postnorm_12', 'scale'] [2048]
['params', 'decoder', 'compose_12', 'mudd_prenorm_12', 'scale'] [2048]
['params', 'decoder', 'compose_13', 'mudd_postnorm_13', 'scale'] [2048]
['params', 'decoder', 'compose_13', 'mudd_prenorm_13', 'scale'] [2048]
['params', 'decoder', 'compose_14', 'mudd_postnorm_14', 'scale'] [2048]
['params', 'decoder', 'compose_14', 'mudd_prenorm_14', 'scale'] [2048]
['param

I0603 07:40:57.037054  122772 google_auth_provider.cc:181] Running on GCE, using service account 626151558586-compute@developer.gserviceaccount.com


In [30]:
# params: mudd_in_layer=False的模型参数； in_layer_params：mudd_in_layer=True的模型参数
def update_key(key):
    if ('mudd_prenorm' in key or 'mudd_postnorm' in key) and 'compose_' in key:
        assert 'compose_' in key
        layer_inx = int(re.findall('compose_(\d+)', key)[0])
        if layer_inx == 35:
            layer_inx -= 1
        new_key = key.replace('params/decoder/', f'params/decoder/layers_{layer_inx + 1}/')
        # print(f'old key: {key} new_key: {new_key}\n')
        
    elif 'mudd_mlp' in key:
        layer_inx = int(re.findall('layers_(\d+)', key)[0])
        new_key = key.replace('sub_0', f'compose_{layer_inx}')
        if layer_inx != 35:
            new_key = new_key.replace(f'layers_{layer_inx}', f'layers_{layer_inx + 1}')
    else:
        new_key = key
    return new_key

In [31]:
# 验证load出来的模型是否正确
updated_params = {}
shapedtype = {}
for k, v in flatten_dict(restored).items():
    new_k = '/'.join(k)
    updated_key = update_key(new_k)
    if new_k != updated_key:
        assert 'mudd_' in new_k
        print(f'old_key:     {new_k}\nupdated_key: {updated_key}\n\n')
    updated_key = tuple(updated_key.split('/'))
    updated_params[updated_key] = jnp.asarray(v, jnp.bfloat16)
    shapedtype[updated_key] = v.shape

old_key:     params/params/decoder/compose_0/mudd_postnorm_0/scale
updated_key: params/params/decoder/layers_1/compose_0/mudd_postnorm_0/scale


old_key:     params/params/decoder/compose_0/mudd_prenorm_0/scale
updated_key: params/params/decoder/layers_1/compose_0/mudd_prenorm_0/scale


old_key:     params/params/decoder/compose_1/mudd_postnorm_1/scale
updated_key: params/params/decoder/layers_2/compose_1/mudd_postnorm_1/scale


old_key:     params/params/decoder/compose_1/mudd_prenorm_1/scale
updated_key: params/params/decoder/layers_2/compose_1/mudd_prenorm_1/scale


old_key:     params/params/decoder/compose_10/mudd_postnorm_10/scale
updated_key: params/params/decoder/layers_11/compose_10/mudd_postnorm_10/scale


old_key:     params/params/decoder/compose_10/mudd_prenorm_10/scale
updated_key: params/params/decoder/layers_11/compose_10/mudd_prenorm_10/scale


old_key:     params/params/decoder/compose_11/mudd_postnorm_11/scale
updated_key: params/params/decoder/layers_12/compose_11/m

In [33]:
# 4.save model
unflatten_convert_params = unflatten_dict(updated_params)
checkpoint_dir = 'gs://newproject-1-llm_base_models_europe-west4/v3.5mini/DreamMiniXL_32k_0530/checkpoints/0/items'
orbax_checkpointer = ocp.PyTreeCheckpointer()
orbax_checkpointer.save(checkpoint_dir, unflatten_convert_params, force=True)
print(f"Quantized params checkpoint saved at: {checkpoint_dir}")

Quantized params checkpoint saved at: gs://newproject-1-llm_base_models_europe-west4/v3.5mini/DreamMiniXL_32k_0530/checkpoints/0/items


In [34]:
# load
mesh_axes = ['data', 'stage', 'fsdp', 'fsdp_transpose', 'sequence', 'tensor', 'tensor_transpose', 'tensor_sequence', 'expert', 'autoregressive']
devices = np.asarray(jax.devices()).reshape([1] * len(mesh_axes))
mesh = jax.sharding.Mesh(devices, mesh_axes)
sharding = jax.sharding.NamedSharding(mesh, PS()) # Sharding is None because we use cpu to load weights
weight_dtype = jnp.bfloat16 # set restore weights dtype, np.float32 or np.float16
abstract_unboxed_params = {}
    
for k, shape in shapedtype.items():
    print(k, shape)
    abstract_unboxed_params[k] = jax.ShapeDtypeStruct(shape=shape, dtype=weight_dtype, sharding=sharding)    
    
abstract_unboxed_params = unflatten_dict(abstract_unboxed_params)['params']

checkpoint_dir = 'gs://newproject-1-llm_base_models_europe-west4/v3.5mini/DreamMiniXL_32k_0530/checkpoints/0/items'

ckpt = epath.Path(checkpoint_dir)
ckptr = ocp.PyTreeCheckpointer()
restore_args = ocp.checkpoint_utils.construct_restore_args(abstract_unboxed_params)
# 如果restored只是一个带有模型名字的字典，没有具体的value矩阵，可以检查下abstract_unboxed_params是不是多了或者漏了params这个key
restored2 = ckptr.restore(
  ckpt, item={'params': abstract_unboxed_params}, transforms={}, restore_args={'params': restore_args}
)

('params', 'params', 'decoder', 'layers_1', 'compose_0', 'mudd_postnorm_0', 'scale') (2048,)
('params', 'params', 'decoder', 'layers_1', 'compose_0', 'mudd_prenorm_0', 'scale') (2048,)
('params', 'params', 'decoder', 'layers_2', 'compose_1', 'mudd_postnorm_1', 'scale') (2048,)
('params', 'params', 'decoder', 'layers_2', 'compose_1', 'mudd_prenorm_1', 'scale') (2048,)
('params', 'params', 'decoder', 'layers_11', 'compose_10', 'mudd_postnorm_10', 'scale') (2048,)
('params', 'params', 'decoder', 'layers_11', 'compose_10', 'mudd_prenorm_10', 'scale') (2048,)
('params', 'params', 'decoder', 'layers_12', 'compose_11', 'mudd_postnorm_11', 'scale') (2048,)
('params', 'params', 'decoder', 'layers_12', 'compose_11', 'mudd_prenorm_11', 'scale') (2048,)
('params', 'params', 'decoder', 'layers_13', 'compose_12', 'mudd_postnorm_12', 'scale') (2048,)
('params', 'params', 'decoder', 'layers_13', 'compose_12', 'mudd_prenorm_12', 'scale') (2048,)
('params', 'params', 'decoder', 'layers_14', 'compose_13'

In [36]:
# for k, v in flatten_dict(restored2).items():
#     print(k, v.shape, v.sum())